# resonus-core: impulse-response measurement

All examples run on the simulated device (`SimulatedIO`), so they work without hardware. Replace it with `SounddeviceIO(48000, input_channels=..., output_channels=...)` for a real measurement.

In [ ]:
import numpy as np

from resonus_core.audio import SimulatedIO
from resonus_core.harmonics import split_harmonics, thd
from resonus_core.measure import MeasurementSettings, measure
from resonus_core.plot import MplPlotView

FS = 48000
# a small 'room': direct sound, two reflections, a decaying tail
rng = np.random.default_rng(0)
t = np.arange(int(0.5 * FS)) / FS
room = 0.05 * np.exp(-t / 0.08) * rng.standard_normal(t.size)
room[[20, 300, 900]] += [1.0, 0.4, -0.25]

## 1. Exponential sweep
One sweep with a silent tail; the result holds causal impulse responses and frequency responses (computed with 0.1 s of pre-ringing, see `resonus_core.measure`).

In [ ]:
settings = MeasurementSettings(duration=2.0, tail=0.6, ir_length=0.5, fade_in=0.05, fade_out=0.005)
result = measure(settings, SimulatedIO(FS, impulse_responses=room, noise_rms=1e-4))
fr = result.frequency_responses[0]
view = MplPlotView()
view.set_axes(
    xlabel="Frequency [Hz]",
    ylabel="Magnitude [dB re 1]",
    xscale="log",
    xlim=(20, 20000),
    title="Transfer function",
)
view.set_trace("sweep", fr.frequency[1:], fr.magnitude_db()[1:], label="Exponential sweep")
view

## 2. Harmonic distortion with a synchronized sweep
A memoryless nonlinearity `x + 0.1 x^2` before the room; the 2nd-harmonic IR is separated (Novák 2015) and THD computed.

In [ ]:
s = MeasurementSettings(
    excitation="sync_sweep",
    level=0.0,
    duration=2.0,
    tail=0.6,
    ir_length=0.5,
    fade_in=0.05,
    fade_out=0.005,
)


class Distorting(SimulatedIO):
    def playrec(self, signal):
        x = np.asarray(signal)
        return super().playrec(x + 0.1 * x**2)


r = measure(s, Distorting(FS, impulse_responses=room[:2048]))
h = split_harmonics(r.deconvolved, r.excitation, n_max=2, length=2048, pre=256)
print(
    "THD at 1 kHz:",
    thd(h, np.array([1000.0]), channel=0)[0].round(4),
    "(theory: 0.05 x |G(2 kHz)|/|G(1 kHz)| of the room)",
)

## 3. Loopback reference channel
Input 1 records the loopback (latency and gain of the sound card), input 2 the microphone. The result excludes the reference and removes its latency.

In [ ]:
latency = 137
h = np.zeros((2, 1, room.size + latency))
h[0, 0, latency:] = room  # io inputs: measurement channels first, then the reference
h[1, 0, latency] = 0.8
r = measure(
    MeasurementSettings(
        duration=2.0,
        tail=0.6,
        ir_length=0.5,
        fade_in=0.05,
        fade_out=0.005,
        reference_channel=1,
        input_channels=(2,),  # device input 1: loopback, input 2: room
    ),
    SimulatedIO(FS, impulse_responses=h),
)
print(
    "channels:",
    len(r.impulse_responses),
    "| direct sound at sample",
    int(np.argmax(np.abs(r.impulse_responses[0].data))),
    "(true 20, latency removed)",
)

## 4. Periodic pink noise
The period (`duration`) sets frequency resolution and the longest IR; `repetitions` sets the averaging. The same works continuously with `resonus_core.continuous.ContinuousAnalyzer`.

In [ ]:
r = measure(
    MeasurementSettings(
        excitation="pink_noise", duration=1.0, level=-20.0, repetitions=4, ir_length=0.5
    ),
    SimulatedIO(FS, impulse_responses=room, noise_rms=1e-3),
)
view.set_trace(
    "pink",
    r.frequency_responses[0].frequency[1:],
    r.frequency_responses[0].magnitude_db()[1:],
    label="Periodic pink noise",
)
view